In [1]:
import os
import pandas as pd
import numpy as np
import pickle as pkl
from rdkit.Chem import MolFromSmiles, MolToInchiKey
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from rdkit.Chem.Descriptors import ExactMolWt
from pathlib import Path
#import mist.utils as utils
from pathlib import Path

ENABLE_FILE_EXPORTS = True

In [2]:
seed = 42
num_mols = 'all'

In [3]:
def write_mgf(df, output_path):
    if ENABLE_FILE_EXPORTS:
        with open(output_path, "w") as f:
            for idx, row in df.iterrows():
                mz_int_pairs = row['spec']
                f.write("BEGIN IONS\n")
                f.write(f"FEATURE_ID={row['compound']}\n")
                f.write(f"PEPMASS={row['parentmass']:.6f}\n")
                f.write("CHARGE=1+\n")
                f.write("MSLEVEL=1\n")
                f.write("IONIZATION_MODE=EI\n")
                f.write(f"TITLE={row['compound']}\n")
                for mz, intensity in mz_int_pairs:
                    f.write(f"{mz} {intensity}\n")
                f.write("END IONS\n\n")
    else:
        print(f'ENABLE_FILE_EXPORTS: {ENABLE_FILE_EXPORTS}')

## Sample of NEIMS gecko

In [4]:
path_to_data = f'../../../data/neims/gecko_EIMS_spectra'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [5]:
fname = f'{path_to_data}/df_neims_gecko+_3_9_22.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)
#df = df.sample(num_mols, random_state=seed)
df

,SMILES,spec
0,C=O,"[[14, 202], [15, 313], [16, 165], [18, 16], [1..."
1,O=C[N+](=O)[O-],"[[14, 72], [16, 151], [17, 70], [18, 108], [19..."
2,O=CC(=O)[N+](=O)[O-],"[[14, 153], [15, 174], [16, 125], [17, 36], [1..."
3,O=C(O)C(=O)[N+](=O)[O-],"[[14, 120], [15, 236], [16, 149], [17, 78], [1..."
4,O=C(OO)C(=O)[N+](=O)[O-],"[[14, 136], [15, 268], [16, 152], [17, 62], [1..."
...,...,...
325118,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,"[[14, 5], [15, 17], [16, 0], [26, 3], [27, 49]..."
325119,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,"[[2, 3], [8, 0], [14, 15], [15, 30], [18, 21],..."
325120,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,"[[14, 18], [15, 44], [16, 1], [18, 1], [20, 5]..."
325121,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,"[[14, 0], [15, 27], [26, 21], [27, 30], [28, 6..."


In [6]:
labels_df = pd.read_csv(f'{path_to_data}/mist_inputs/labels.tsv', sep='\t')
labels_df['compound'] = labels_df['spec']
labels_df = labels_df.drop(columns=['smiles', 'Unnamed: 0', 'spec'])
labels_df

,dataset,ionization,formula,inchikey,instrument,compound
0,neims,[M]+,CH2O,WSFSSNUMVMOOMR-UHFFFAOYSA-N,simulated,gecko_0
1,neims,[M]+,CHNO3,CAMJPMYWRKKZNK-UHFFFAOYSA-N,simulated,gecko_1
2,neims,[M]+,C2HNO4,UJDBJZXDOVXPFV-UHFFFAOYSA-N,simulated,gecko_2
3,neims,[M]+,C2HNO5,RMULBIRLFFXIEL-UHFFFAOYSA-N,simulated,gecko_3
4,neims,[M]+,C2HNO6,NVYYGCURLZHQKD-UHFFFAOYSA-N,simulated,gecko_4
...,...,...,...,...,...,...
325118,neims,[M]+,C16H32N2O15Si3,JHFUNAOALBCUEU-UHFFFAOYSA-N,simulated,gecko_tms_166429
325119,neims,[M]+,C16H33NO11Si3,WIPWTSILCDGNDN-UHFFFAOYSA-N,simulated,gecko_tms_166430
325120,neims,[M]+,C13H23NO11Si2,TZYJPJSYLIRUID-UHFFFAOYSA-N,simulated,gecko_tms_166431
325121,neims,[M]+,C12H22N2O14Si2,FAZHSBRCQRQGLZ-UHFFFAOYSA-N,simulated,gecko_tms_166432


In [7]:
df = pd.merge(df, labels_df, left_index=True, right_index=True)
df

,SMILES,spec,dataset,ionization,formula,inchikey,instrument,compound
0,C=O,"[[14, 202], [15, 313], [16, 165], [18, 16], [1...",neims,[M]+,CH2O,WSFSSNUMVMOOMR-UHFFFAOYSA-N,simulated,gecko_0
1,O=C[N+](=O)[O-],"[[14, 72], [16, 151], [17, 70], [18, 108], [19...",neims,[M]+,CHNO3,CAMJPMYWRKKZNK-UHFFFAOYSA-N,simulated,gecko_1
2,O=CC(=O)[N+](=O)[O-],"[[14, 153], [15, 174], [16, 125], [17, 36], [1...",neims,[M]+,C2HNO4,UJDBJZXDOVXPFV-UHFFFAOYSA-N,simulated,gecko_2
3,O=C(O)C(=O)[N+](=O)[O-],"[[14, 120], [15, 236], [16, 149], [17, 78], [1...",neims,[M]+,C2HNO5,RMULBIRLFFXIEL-UHFFFAOYSA-N,simulated,gecko_3
4,O=C(OO)C(=O)[N+](=O)[O-],"[[14, 136], [15, 268], [16, 152], [17, 62], [1...",neims,[M]+,C2HNO6,NVYYGCURLZHQKD-UHFFFAOYSA-N,simulated,gecko_4
...,...,...,...,...,...,...,...,...
325118,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,"[[14, 5], [15, 17], [16, 0], [26, 3], [27, 49]...",neims,[M]+,C16H32N2O15Si3,JHFUNAOALBCUEU-UHFFFAOYSA-N,simulated,gecko_tms_166429
325119,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,"[[2, 3], [8, 0], [14, 15], [15, 30], [18, 21],...",neims,[M]+,C16H33NO11Si3,WIPWTSILCDGNDN-UHFFFAOYSA-N,simulated,gecko_tms_166430
325120,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,"[[14, 18], [15, 44], [16, 1], [18, 1], [20, 5]...",neims,[M]+,C13H23NO11Si2,TZYJPJSYLIRUID-UHFFFAOYSA-N,simulated,gecko_tms_166431
325121,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,"[[14, 0], [15, 27], [26, 21], [27, 30], [28, 6...",neims,[M]+,C12H22N2O14Si2,FAZHSBRCQRQGLZ-UHFFFAOYSA-N,simulated,gecko_tms_166432


In [8]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['parentmass'] = df['mols'].apply(ExactMolWt)

In [10]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'neims'
df['compound'] = 'neims_' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['inchikey'] = df['mols'].apply(MolToInchiKey)
df['instrument'] = 'simulated'

In [9]:
df

,SMILES,spec,dataset,ionization,formula,inchikey,instrument,compound,mols,parentmass
0,C=O,"[[14, 202], [15, 313], [16, 165], [18, 16], [1...",neims,[M]+,CH2O,WSFSSNUMVMOOMR-UHFFFAOYSA-N,simulated,gecko_0,<rdkit.Chem.rdchem.Mol object at 0x7fde98d2cdd0>,30.010565
1,O=C[N+](=O)[O-],"[[14, 72], [16, 151], [17, 70], [18, 108], [19...",neims,[M]+,CHNO3,CAMJPMYWRKKZNK-UHFFFAOYSA-N,simulated,gecko_1,<rdkit.Chem.rdchem.Mol object at 0x7fde98d2c9e0>,74.995643
2,O=CC(=O)[N+](=O)[O-],"[[14, 153], [15, 174], [16, 125], [17, 36], [1...",neims,[M]+,C2HNO4,UJDBJZXDOVXPFV-UHFFFAOYSA-N,simulated,gecko_2,<rdkit.Chem.rdchem.Mol object at 0x7fde98d2cd60>,102.990558
3,O=C(O)C(=O)[N+](=O)[O-],"[[14, 120], [15, 236], [16, 149], [17, 78], [1...",neims,[M]+,C2HNO5,RMULBIRLFFXIEL-UHFFFAOYSA-N,simulated,gecko_3,<rdkit.Chem.rdchem.Mol object at 0x7fde98d2cc10>,118.985472
4,O=C(OO)C(=O)[N+](=O)[O-],"[[14, 136], [15, 268], [16, 152], [17, 62], [1...",neims,[M]+,C2HNO6,NVYYGCURLZHQKD-UHFFFAOYSA-N,simulated,gecko_4,<rdkit.Chem.rdchem.Mol object at 0x7fde98d2cb30>,134.980387
...,...,...,...,...,...,...,...,...,...,...
325118,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,"[[14, 5], [15, 17], [16, 0], [26, 3], [27, 49]...",neims,[M]+,C16H32N2O15Si3,JHFUNAOALBCUEU-UHFFFAOYSA-N,simulated,gecko_tms_166429,<rdkit.Chem.rdchem.Mol object at 0x7fde8ba6ccf0>,576.111048
325119,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,"[[2, 3], [8, 0], [14, 15], [15, 30], [18, 21],...",neims,[M]+,C16H33NO11Si3,WIPWTSILCDGNDN-UHFFFAOYSA-N,simulated,gecko_tms_166430,<rdkit.Chem.rdchem.Mol object at 0x7fde8ba6cd60>,499.136140
325120,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,"[[14, 18], [15, 44], [16, 1], [18, 1], [20, 5]...",neims,[M]+,C13H23NO11Si2,TZYJPJSYLIRUID-UHFFFAOYSA-N,simulated,gecko_tms_166431,<rdkit.Chem.rdchem.Mol object at 0x7fde8ba6cdd0>,425.080964
325121,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,"[[14, 0], [15, 27], [26, 21], [27, 30], [28, 6...",neims,[M]+,C12H22N2O14Si2,FAZHSBRCQRQGLZ-UHFFFAOYSA-N,simulated,gecko_tms_166432,<rdkit.Chem.rdchem.Mol object at 0x7fde8ba6ce40>,474.060956


#### Write spec to MGF file

In [10]:
output_path = Path(f'{output_dir}/neims_gecko+_spectra_{num_mols}.mgf')
write_mgf(df, output_path)

#### Write labels to .tsv

In [13]:
neims_labels = df[['dataset', 'compound', 'ionization', 'formula', 'SMILES', 'inchikey', 'instrument']]
neims_labels.index = df['compound']
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_gecko_labels_{num_mols}.tsv', sep='\t')
    df['SMILES'].to_csv(f'{output_dir}/neims_gecko_smiles_{num_mols}.csv')
    df['SMILES'].to_csv(f'{output_dir}/neims_gecko_lookup_smiles_{num_mols}.txt', header=None, index=False)

In [10]:
raise Exception

Exception: 

## Load canopus aug data

In [ ]:
path_to_data = f'../../../data/paired_spectra/canopus_train/aug_iceberg_canopus_train/canopus_hplus_100_0'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
canopus_data = utils.parse_spectra_mgf(f'{path_to_data}/full_out.mgf')

400000it [00:25, 15429.23it/s]


In [ ]:
canopus_subset = canopus_data[0:num_mols]
subset_str = utils.build_mgf_str(canopus_subset)
#np.savetxt('../data/paired_spectra/canopus_train/aug_iceberg_canopus_train/canopus_hplus_100_0/subset_out.mgf', subset_str)
if ENABLE_FILE_EXPORTS:
    with open(f'{output_dir}/aug_canopus_spectra_{num_mols}.mgf', 'w+') as file:
        file.writelines(subset_str)
subset_ids = []
for idx, el in enumerate(canopus_subset):
    subset_ids.append(el[0]['ID'])
subset_ids[0:5]

100%|██████████| 10000/10000 [00:01<00:00, 5310.60it/s]


['aug_89166', 'aug_72305', 'aug_70314', 'aug_137643', 'aug_19834']

In [ ]:
canopus_labels = pd.read_csv(f'../../../data/paired_spectra/canopus_train/aug_iceberg_canopus_train/biomols_filtered_smiles_canopus_train_labels.tsv', sep='\t')
canopus_labels = canopus_labels[canopus_labels['spec'].isin(subset_ids)]
ENABLE_FILE_EXPORTS= True
if ENABLE_FILE_EXPORTS:
    canopus_labels.to_csv(f'{output_dir}/aug_canopus_labels_{num_mols}.tsv', sep='\t')

In [ ]:
df_out = pd.DataFrame()
df_out['SMILES'] = canopus_labels['smiles'].values
if ENABLE_FILE_EXPORTS:
    df_out.to_csv(f'{output_dir}/aug_canopus_smiles_{num_mols}.csv')

## Exp canopus

In [ ]:
path_to_data = f'../../../data/paired_spectra/canopus_train/'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/exp_canopus_train/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
files = ['../../../data/paired_spectra/canopus_train/splits/canopus_hplus_100_0.tsv',
         '../../../data/paired_spectra/canopus_train/splits/canopus_hplus_100_1.tsv',
         '../../../data/paired_spectra/canopus_train/splits/canopus_hplus_100_2.tsv']
dfs = []
for file in files:
    split = pd.read_csv(file, sep='\t')
    test_split = split[split['split'] == 'test']
    dfs.append(test_split)
splits = pd.concat(dfs)
splits_all_test = splits.drop_duplicates(subset='name').drop(columns=['split'])
if ENABLE_FILE_EXPORTS:
    splits_all_test.to_csv(f'{output_dir}/exp_canopus_all_test_labels.tsv', sep='\t', index=False)

In [ ]:
tests = splits_all_test['name'].values
tests
specs = []
for measurement in tests:
    file = f'{path_to_data}/spec_files/{measurement}.ms'
    spec = utils.parse_spectra(file)
    specs.append(spec)
tests.shape

(2206,)

In [ ]:
mgf_str = utils.build_mgf_str(specs)

100%|██████████| 2206/2206 [00:04<00:00, 520.63it/s]


In [ ]:
ENABLE_FILE_EXPORTS = True
if ENABLE_FILE_EXPORTS:
    with open(f'{output_dir}/exp_canopus_all_test_spectra.mgf', 'w+') as file:
        file.writelines(mgf_str)

In [ ]:
labels = pd.read_csv(f'{path_to_data}/labels.tsv', sep='\t')
labels = labels[labels['spec'].isin(tests)]
if ENABLE_FILE_EXPORTS:
    labels.to_csv(f'{output_dir}/exp_canopus_all_test_labels.tsv', sep='\t')

In [ ]:
df_out = pd.DataFrame()
df_out['SMILES'] = labels['smiles'].values
ENABLE_FILE_EXPORTS = True
if ENABLE_FILE_EXPORTS:
    df_out.to_csv(f'{output_dir}/exp_canopus_all_test_smiles.csv')

### Franklin neims

In [ ]:
path_to_data = f'../../../data/neims/Franklin_dataset'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
fname = f'{path_to_data}/df_neims_franklin_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'neims_fr'
df['compound'] = 'neims_fr' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'simulated'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CC(C)=CCC/C(C)=C/CC/C(C)=C/CCC(=C)C=C,"[[26, 27], [27, 154], [28, 56], [29, 213], [30...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab9a0>,C20H32,[M]+,neims_fr,neims_fr0,272.250401,simulated
1,CC(=O)C=CC1(C)C(=O)C2(CCC1(C)C)CO2,"[[25, 5], [26, 55], [27, 225], [28, 113], [29,...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab820>,C14H20O3,[M]+,neims_fr,neims_fr1,236.141244,simulated
2,CC1(C)CCC[C@@]2(C)OC(=O)C=C12,"[[14, 2], [15, 13], [26, 102], [27, 418], [28,...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab760>,C11H16O2,[M]+,neims_fr,neims_fr2,180.115030,simulated
3,NC1=CC2=C(C=C1)NC3=C2C=CC=C3,"[[24, 2], [26, 5], [27, 39], [28, 29], [37, 27...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab460>,C12H10N2,[M]+,neims_fr,neims_fr3,182.084398,simulated
4,CC(C)CCCC(C)CCCC1(C)CCC(=O)O1,"[[29, 42], [39, 49], [41, 447], [42, 182], [43...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab5e0>,C16H30O2,[M]+,neims_fr,neims_fr4,254.224580,simulated
...,...,...,...,...,...,...,...,...,...
64,C[Si](C)(C)OC(=O)C1CCC(=O)N1,"[[14, 8], [15, 68], [16, 3], [18, 32], [26, 63...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610e20>,C8H15NO3Si,[M]+,neims_fr,neims_fr64,201.082120,simulated
65,CCCOC(=O)C1=CC=CC=C1C(=O)OCC(C)CC,"[[27, 96], [28, 10], [29, 111], [39, 69], [41,...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610b80>,C16H22O4,[M]+,neims_fr,neims_fr65,278.151809,simulated
66,C[Si](C)(C)OC(=O)C1=CC=CC=C1C(=O)O[Si](C)(C)C,"[[14, 1], [15, 86], [18, 0], [29, 112], [30, 0...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610ca0>,C14H22O4Si2,[M]+,neims_fr,neims_fr66,310.105662,simulated
67,COC1=CC(C(CO[Si](C)(C)C)O[Si](C)(C)C)=CC=C1O[S...,"[[29, 18], [41, 10], [43, 59], [44, 71], [45, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610b20>,C18H36O4Si3,[M]+,neims_fr,neims_fr67,400.192139,simulated


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_franklin_labels.tsv', sep='\t')

In [ ]:
output_path = Path(f'{output_dir}/neims_franklin_spectra.mgf')
write_mgf(df, output_path)

### Franklin exp

In [ ]:
fname = f'{path_to_data}/df_exp_franklin_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'exp_fr'
df['compound'] = 'exp_fr' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'unknown'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CC(C)=CCC/C(C)=C/CC/C(C)=C/CCC(=C)C=C,"[[41, 543], [42, 29], [43, 80], [53, 76], [54,...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c537c0>,C20H32,[M]+,exp_fr,exp_fr0,272.250401,unknown
1,CC(=O)C=CC1(C)C(=O)C2(CCC1(C)C)CO2,"[[26, 7], [27, 94], [28, 23], [29, 67], [30, 3...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c53ac0>,C14H20O3,[M]+,exp_fr,exp_fr1,236.141244,unknown
2,CC1(C)CCC[C@@]2(C)OC(=O)C=C12,"[[38, 3], [39, 59], [40, 19], [41, 106], [42, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c53700>,C11H16O2,[M]+,exp_fr,exp_fr2,180.115030,unknown
3,NC1=CC2=C(C=C1)NC3=C2C=CC=C3,"[[27, 5], [28, 14], [38, 3], [39, 10], [41, 4]...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c535e0>,C12H10N2,[M]+,exp_fr,exp_fr3,182.084398,unknown
4,CC(C)CCCC(C)CCCC1(C)CCC(=O)O1,"[[39, 31], [40, 6], [41, 175], [42, 43], [43, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c53580>,C16H30O2,[M]+,exp_fr,exp_fr4,254.224580,unknown
...,...,...,...,...,...,...,...,...,...
64,C[Si](C)(C)OC(=O)C1CCC(=O)N1,"[[15, 2], [26, 3], [27, 13], [28, 72], [29, 15...",<rdkit.Chem.rdchem.Mol object at 0x7efde3606d00>,C8H15NO3Si,[M]+,exp_fr,exp_fr64,201.082120,unknown
65,CCCOC(=O)C1=CC=CC=C1C(=O)OCC(C)CC,"[[27, 10], [29, 11], [39, 7], [41, 28], [42, 1...",<rdkit.Chem.rdchem.Mol object at 0x7efde3606b80>,C16H22O4,[M]+,exp_fr,exp_fr65,278.151809,unknown
66,C[Si](C)(C)OC(=O)C1=CC=CC=C1C(=O)O[Si](C)(C)C,"[[25, 2], [26, 6], [27, 4], [29, 2], [31, 1], ...",<rdkit.Chem.rdchem.Mol object at 0x7efde36069a0>,C14H22O4Si2,[M]+,exp_fr,exp_fr66,310.105662,unknown
67,COC1=CC(C(CO[Si](C)(C)C)O[Si](C)(C)C)=CC=C1O[S...,"[[40, 17], [43, 13], [44, 19], [45, 94], [46, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde3606b20>,C18H36O4Si3,[M]+,exp_fr,exp_fr67,400.192139,unknown


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/exp_franklin_labels.tsv', sep='\t')

In [ ]:
output_path = Path(f'{output_dir}/exp_franklin_spectra.mgf')
write_mgf(df, output_path)

## canopus neims

In [ ]:
path_to_data = f'../../../data/neims/mist'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
fname = f'{path_to_data}/df_neims_canopus_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'canopus_neims'
df['compound'] = 'canopus_neims_' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'simulated'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CCC(=O)CCCCCC1NC(=O)C2CCCCN2C(=O)C(Cc2ccccc2)N...,"[[28, 9], [30, 3], [32, 55], [36, 97], [40, 28...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e69a0>,C28H40N4O5,[M]+,canopus_neims,canopus_neims_0,512.299870,simulated
1,CCCCCCCCCCCCCC(=O)NC(C)(C)C(=O)N1CCCC1C(=O)NC(...,"[[28, 7], [30, 54], [39, 16], [43, 69], [44, 3...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6ac0>,C45H83N7O8,[M]+,canopus_neims,canopus_neims_1,849.630313,simulated
2,CC(C)CC1NC(=O)C(NC(=O)c2ncccc2O)C(C)OC(=O)C(c2...,"[[30, 43], [39, 338], [40, 159], [41, 761], [4...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6a00>,C44H62N8O11,[M]+,canopus_neims,canopus_neims_2,878.453805,simulated
3,COc1cc(C2C3(O)C(O)C4CC2(O)C(O)(C(=O)O4)C3C(=O)...,"[[25, 1], [26, 30], [27, 65], [28, 104], [29, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6b20>,C22H20O10,[M]+,canopus_neims,canopus_neims_3,444.105647,simulated
4,COC1C=COC2(C)Oc3c(C)c(O)c4c(c3C2=O)C(=O)C=C(NC...,"[[25, 9], [30, 21], [32, 15], [33, 9], [38, 7]...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6a60>,C37H45NO12,[M]+,canopus_neims,canopus_neims_4,695.294176,simulated
...,...,...,...,...,...,...,...,...,...
2201,CCOc1ccc2oc(=O)c(-c3ccc(O)cc3)c(C)c2c1,"[[0, 0], [14, 12], [15, 47], [16, 4], [17, 2],...",<rdkit.Chem.rdchem.Mol object at 0x7efde33abac0>,C18H16O4,[M]+,canopus_neims,canopus_neims_2201,296.104859,simulated
2202,Cc1cn(C2OC(CO)C(O)C2O)c(=O)nc1N,"[[14, 38], [15, 122], [16, 32], [17, 52], [18,...",<rdkit.Chem.rdchem.Mol object at 0x7efde33aba60>,C10H15N3O5,[M]+,canopus_neims,canopus_neims_2202,257.101171,simulated
2203,COc1cc(OC)c(C(=O)C=Cc2ccc(OC)c(OC)c2)c(OC)c1,"[[14, 2], [15, 162], [26, 46], [27, 123], [28,...",<rdkit.Chem.rdchem.Mol object at 0x7efde33abb20>,C20H22O6,[M]+,canopus_neims,canopus_neims_2203,358.141638,simulated
2204,O=C1NCCc2c1[nH]c1c(Cl)cccc21,"[[33, 2], [35, 16], [36, 43], [37, 37], [38, 7...",<rdkit.Chem.rdchem.Mol object at 0x7efde33abbe0>,C11H9ClN2O,[M]+,canopus_neims,canopus_neims_2204,220.040341,simulated


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_canopus_labels.tsv', sep='\t')

output_path = Path(f'{output_dir}/neims_canopus_spectra.mgf')
write_mgf(df, output_path)

## canopus aug neims

In [ ]:
fname = f'{path_to_data}/df_neims_aug_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'aug_neims'
df['compound'] = 'aug_neims_' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'simulated'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CCCCCCCCCCCCCCCCOCC(COP(=O)([O-])OCC[N+](C)(C)...,"[[25, 24], [26, 109], [27, 238], [28, 168], [2...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff73880>,C46H82NO7P,[M]+,aug_neims,aug_neims_0,791.582891,simulated
1,CC1=C(C(=O)OC2=C1C=CC(=C2)OC(C)C(=O)NCCC(=O)O)C,"[[29, 67], [33, 58], [34, 27], [36, 32], [38, ...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff738e0>,C17H19NO6,[M]+,aug_neims,aug_neims_1,333.121237,simulated
2,CC(C)C1=CC=C(C=C1)CN2CCC(C2)N(C)S(=O)(=O)C3=CC...,"[[15, 6], [27, 5], [28, 98], [30, 260], [33, 5...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff73940>,C22H30N2O4S2,[M]+,aug_neims,aug_neims_2,450.164699,simulated
3,CCN1C2=C(C=C(C=C2)S(=O)(=O)N(C)C)N=C1CCC(=O)NC...,"[[33, 55], [34, 75], [36, 188], [37, 77], [38,...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff739a0>,C20H23N5O5S,[M]+,aug_neims,aug_neims_3,445.141990,simulated
4,CCC(C(=O)NC1=C(C=CC(=C1)Cl)Cl)N(C2=CC=CC=C2)S(...,"[[33, 32], [34, 1], [35, 98], [36, 149], [37, ...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff73a00>,C17H18Cl2N2O3S,[M]+,aug_neims,aug_neims_4,400.041519,simulated
...,...,...,...,...,...,...,...,...,...
9995,CN(C)CCN1C=C(C=N1)NC(=O)CCCC2=NC(=NO2)C3=CC=C(...,"[[33, 8], [36, 45], [37, 20], [38, 39], [39, 9...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb956a0>,C19H23ClN6O2,[M]+,aug_neims,aug_neims_9995,402.157102,simulated
9996,CC(C(=O)NC(CC1=CN=CN1)C(=O)O)NC(=O)C(CCCN=C(N)...,"[[16, 72], [17, 72], [18, 74], [25, 0], [28, 5...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb95700>,C24H35N9O6,[M]+,aug_neims,aug_neims_9996,545.271030,simulated
9997,CCC(C(=O)OC)SC1=NN=C(S1)NC(=O)C2=NOC(=C2)C3=CC...,"[[29, 64], [33, 105], [34, 51], [35, 20], [36,...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb95760>,C15H14N4O5S2,[M]+,aug_neims,aug_neims_9997,394.040562,simulated
9998,CC1(C2CC=C(C1C2)C(=O)O)C,"[[14, 5], [16, 13], [17, 5], [26, 102], [27, 4...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb957c0>,C10H14O2,[M]+,aug_neims,aug_neims_9998,166.099380,simulated


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'    
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_aug_labels.tsv', sep='\t')

output_path = Path(f'{output_dir}/neims_aug_spectra.mgf')
write_mgf(df, output_path)